# Churn Prediction

A full implementation of a multilayer neural network built entirely in NumPy, without any deep learning frameworks. This notebook covers every component of the network by hand — forward propagation, backpropagation, He initialization, mini-batch gradient descent, and binary cross-entropy loss — applied to a binary churn classification problem. 

The final architecture is a 4-layer network (128→64→32→1) trained with ReLU activations in the hidden layers and a sigmoid output 

Key implementation details:

* He initialization for stable gradient flow through ReLU layers
* Mini-batch gradient descent with batch size 512 for faster convergence
* Numerical stability via log-clipping in the loss function
* Full preprocessing pipeline including one-hot encoding and feature normalization using training set statistics

This was built as a portfolio project to develop a deeper understanding of how neural networks work under the hood, before relying on higher-level frameworks like PyTorch or TensorFlow.

In [1]:
!pip install imblearn

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
# Import required libraries for machine learning 
from sklearn.model_selection import train_test_split, cross_val_score,StratifiedKFold, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, 
                confusion_matrix, roc_auc_score, 
                roc_curve,precision_recall_curve, 
                average_precision_score, f1_score,precision_score, 
                recall_score, accuracy_score)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import xgboost as xgb
import lightgbm as lgb
import joblib
import warnings
warnings.filterwarnings('ignore')

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e3/train.csv
/kaggle/input/competitions/playground-series-s6e3/test.csv


In [3]:
def load_data():
    train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/train.csv")
    test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/test.csv")

    return train, test

In [4]:
def get_info(df):
    return df.info()

In [5]:
def preprocess(df):
    train_df = df.copy()
    train_df = pd.get_dummies(train_df, drop_first=True)
    
    # separate features and target
    X = train_df.drop(["Churn_Yes", "id"], axis=1)
    y = train_df['Churn_Yes']

    # convert boolean cols to int
    bool_cols = X.select_dtypes(include=['bool']).columns
    X[bool_cols] = X[bool_cols].astype(int)

    return X, y

In [6]:
def split_data(X, y):
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, 
        y,
        test_size = 0.2,
        random_state = 42,
        stratify = y
    )
    
    X_train = X_train.reset_index(drop=True)
    X_test = X_test.reset_index(drop=True)
    y_train = y_train.reset_index(drop=True)
    y_test = y_test.reset_index(drop=True)

    return X_train, X_test, y_train, y_test

In [7]:
def matrix_transformation(X_train, X_test, y_train, y_test):
    # Do all of these together for both splits
    X_train = np.array(X_train).T
    X_test = np.array(X_test).T
    
    y_train = np.array(y_train).reshape(1, -1)
    y_test = np.array(y_test).reshape(1, -1)
    
    # Normalize both using X_train stats only
    mean = np.mean(X_train, axis=1, keepdims=True)
    std = np.std(X_train, axis=1, keepdims=True) + 1e-8
    
    # Normalized features
    X_train = (X_train - mean) / std
    X_test = (X_test - mean) / std  # use X_train mean/std, not X_test's own

    return X_train, X_test, y_train, y_test, mean, std

In [8]:
# Initialize weights and biases
def he_initializer(size):
    return np.sqrt(2 / size)
def initialize_params(input_size, hidden_size_1, hidden_size_2, hidden_size_3, output_size):
    np.random.seed(1) # for reproducibility 
   
    weights = {
        'W1': np.random.randn(hidden_size_1, input_size) * he_initializer(input_size),
        'b1': np.zeros((hidden_size_1, 1)),
        'W2': np.random.randn(hidden_size_2, hidden_size_1) * he_initializer(hidden_size_1),
        'b2': np.zeros((hidden_size_2, 1)),
        'W3': np.random.randn(hidden_size_3, hidden_size_2) * he_initializer(hidden_size_2),
        'b3': np.zeros((hidden_size_3, 1)),
        'W4': np.random.randn(output_size, hidden_size_3) * he_initializer(hidden_size_3),
        'b4': np.zeros((output_size, 1))
    }
    return weights

In [9]:
# sigmoid activation function
def sigmoid(z):
    return 1/ (1 + np.exp(-z))

# ReLu activation function
def ReLU(z):
    return np.maximum(0, z)


In [10]:
def dropout(A, keep_prob=0.8):
    mask = np.random.rand(*A.shape) < keep_prob
    return (A * mask) / keep_prob  # divide by keep_prob to keep expected values the same

In [11]:
# Forward propagation
def forward_propagation(X, weights):
    # Extract weights and biases
    W1, b1 = weights['W1'], weights['b1']
    W2, b2 = weights['W2'], weights['b2'],
    W3, b3 = weights['W3'], weights['b3'],
    W4, b4 = weights['W4'], weights['b4']

    # Forward Pass
    Z1 = np.dot(W1, X) + b1
    A1 = ReLU(Z1)
    A1 = dropout(A1, keep_prob=0.8)  # drop 20% of neurons
    Z2 = np.dot(W2, A1) + b2
    A2 = ReLU(Z2)
    A2 = dropout(A2, keep_prob=0.8)  # drop 20% of neurons
    Z3 = np.dot(W3, A2) + b3
    A3 = ReLU(Z3)
    A3 = dropout(A3, keep_prob=0.8)  # drop 20% of neurons
    Z4 = np.dot(W4, A3) + b4
    A4 = sigmoid(Z4)

    # Store values for back propagation
    cache = (Z1, A1, W1, b1, Z2, A2, W2, b2, Z3, A3, W3, b3, Z4, A4, W4, b4)

    return A4, cache

In [12]:
def compute_loss(Y, A4):
    m = Y.shape[1]  # Number of examples
    epsilon = 1e-8
    A4 = np.clip(A4, epsilon, 1 - epsilon)  # keeps A2 away from 0 and 1
    loss = -1/m * np.sum(Y * np.log(A4) + (1 - Y) * np.log(1 - A4))
    return loss

$$\mathcal{L} = -\frac{1}{m} \sum_{i=1}^{m} \Big[ y^{(i)} \log(\hat{y}^{(i)}) + (1 - y^{(i)}) \log(1 - \hat{y}^{(i)}) \Big]$$

In [13]:
def backward_propagation(X, Y, cache):
    (Z1, A1, W1, b1, Z2, A2, W2, b2, Z3, A3, W3, b3, Z4, A4, W4, b4) = cache
    m = X.shape[1]  # note: shape[1] not shape[0] since X is (features, m)

    # Output layer (layer 4)
    dZ4 = A4 - Y
    dW4 = 1/m * np.dot(dZ4, A3.T)  # was A1.T — should be previous layer's activation
    db4 = 1/m * np.sum(dZ4, axis=1, keepdims=True)

    # Layer 3
    dA3 = np.dot(W4.T, dZ4)         # was W2 — should be W4
    dZ3 = dA3 * (A3 > 0)
    dW3 = 1/m * np.dot(dZ3, A2.T)
    db3 = 1/m * np.sum(dZ3, axis=1, keepdims=True)

    # Layer 2
    dA2 = np.dot(W3.T, dZ3)
    dZ2 = dA2 * (A2 > 0)
    dW2 = 1/m * np.dot(dZ2, A1.T)
    db2 = 1/m * np.sum(dZ2, axis=1, keepdims=True)

    # Layer 1
    dA1 = np.dot(W2.T, dZ2)
    dZ1 = dA1 * (A1 > 0)
    dW1 = 1/m * np.dot(dZ1, X.T)
    db1 = 1/m * np.sum(dZ1, axis=1, keepdims=True)

    gradients = {
        'dW1': dW1, 'db1': db1,
        'dW2': dW2, 'db2': db2,
        'dW3': dW3, 'db3': db3,
        'dW4': dW4, 'db4': db4
    }

    return gradients

In [14]:
def update_parameters(weights, gradients, learning_rate):
    # Update weights and biases
    weights['W1'] -= learning_rate * gradients['dW1']
    weights['b1'] -= learning_rate * gradients['db1']
    weights['W2'] -= learning_rate * gradients['dW2']
    weights['b2'] -= learning_rate * gradients['db2']
    weights['W3'] -= learning_rate * gradients['dW3']
    weights['b3'] -= learning_rate * gradients['db3']
    weights['W4'] -= learning_rate * gradients['dW4']
    weights['b4'] -= learning_rate * gradients['db4']
    
    return weights

In [15]:
# learning rate schedule
def get_learning_rate(initial_lr, epoch, decay=0.001):
    return initial_lr / (1 + decay * epoch)

In [16]:
def train_model(X, Y, X_test, y_test, input_size, hidden_size_1, hidden_size_2,
                hidden_size_3, output_size, weights, epochs, learning_rate):
    
    Y = np.array(Y).reshape(1, -1)
    m = X.shape[1]
    batch_size = 512

    for epoch in range(epochs):
        epoch_loss = 0
        num_batches = 0

        # Shuffle batches each epoch
        permutation = np.random.permutation(m)
        X_shuffled = X[:, permutation]
        Y_shuffled = Y[:, permutation]

        lr = get_learning_rate(learning_rate, epoch)

        for i in range(0, m, batch_size):
            X_batch = X_shuffled[:, i:i+batch_size]
            Y_batch = Y_shuffled[:, i:i+batch_size]

            A4, cache = forward_propagation(X_batch, weights)
            loss = compute_loss(Y_batch, A4)
            gradients = backward_propagation(X_batch, Y_batch, cache)
            weights = update_parameters(weights, gradients, lr)

            epoch_loss += loss
            num_batches += 1

        if epoch % 100 == 0:
            A4_val, _ = forward_propagation(X_test, weights)
            val_loss = compute_loss(y_test, A4_val)
            print(f"Epoch {epoch} | Train: {epoch_loss / num_batches:.4f} | Val: {val_loss:.4f}")

    return weights

In [17]:
def predict_proba(X, weights):
    A2, _ = forward_propagation(X, weights)
    return A2  # raw probabilities, shape (1, m)

In [18]:
def test_transformation(test, mean, std):
    test_features = test.drop(columns=['id'])
    
    # Step 1: get_dummies — same as training
    test_encoded = pd.get_dummies(test_features)
    
    # Step 2: align columns with training data
    feature_columns = X.columns
    test_encoded = test_encoded.reindex(columns=feature_columns, fill_value=0)
    
    # Step 3: convert boolean cols to int — same as training
    bool_cols = test_encoded.select_dtypes(include=['bool']).columns
    test_encoded[bool_cols] = test_encoded[bool_cols].astype(int)
    
    # Step 4: convert, transpose, normalize
    X_kaggle_test = np.array(test_encoded).T
    X_kaggle_test = (X_kaggle_test - mean) / std

    return X_kaggle_test

In [19]:
# load data 
train, test = load_data()

# get info
get_info(train)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 594194 entries, 0 to 594193
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                594194 non-null  int64  
 1   gender            594194 non-null  object 
 2   SeniorCitizen     594194 non-null  int64  
 3   Partner           594194 non-null  object 
 4   Dependents        594194 non-null  object 
 5   tenure            594194 non-null  int64  
 6   PhoneService      594194 non-null  object 
 7   MultipleLines     594194 non-null  object 
 8   InternetService   594194 non-null  object 
 9   OnlineSecurity    594194 non-null  object 
 10  OnlineBackup      594194 non-null  object 
 11  DeviceProtection  594194 non-null  object 
 12  TechSupport       594194 non-null  object 
 13  StreamingTV       594194 non-null  object 
 14  StreamingMovies   594194 non-null  object 
 15  Contract          594194 non-null  object 
 16  PaperlessBilling  59

In [20]:
# Preprocess
X, y = preprocess(train)

In [21]:
# Split data
X_train, X_test, y_train, y_test = split_data(X, y)

In [22]:
oversample = SMOTE()
X_train, y_train = oversample.fit_resample(X_train, y_train)

In [23]:
# matrix transformations
X_train, X_test, y_train, y_test, mean, std = matrix_transformation(X_train, X_test, y_train, y_test)

In [24]:
# Define the network structure
input_size = X_train.shape[0]  # Number of input neurons
hidden_size_1 = 128  # Number of hidden neurons
hidden_size_2 = 64
hidden_size_3 = 32
output_size = 1  # Number of output neurons

In [25]:
# Initialize parameters
weights = initialize_params(input_size, hidden_size_1, hidden_size_2, hidden_size_3, output_size)

In [26]:
# Train the network
trained_weights = train_model(X_train, y_train, X_test, y_test, input_size, 
                        hidden_size_1, hidden_size_2, 
                        hidden_size_3, output_size, weights,
                        epochs=1000, learning_rate=0.01)

Epoch 0 | Train: 0.4417 | Val: 0.4356
Epoch 100 | Train: 0.3188 | Val: 0.3766
Epoch 200 | Train: 0.3035 | Val: 0.3654
Epoch 300 | Train: 0.2982 | Val: 0.3637
Epoch 400 | Train: 0.2952 | Val: 0.3596
Epoch 500 | Train: 0.2935 | Val: 0.3601
Epoch 600 | Train: 0.2918 | Val: 0.3594
Epoch 700 | Train: 0.2908 | Val: 0.3584
Epoch 800 | Train: 0.2895 | Val: 0.3576
Epoch 900 | Train: 0.2889 | Val: 0.3582


In [27]:
# Normalize both using X_train stats only
X_kaggle_test = test_transformation(test, mean, std)

In [28]:
# Predict and submit
probabilities = predict_proba(X_kaggle_test, trained_weights)

submission_df = pd.DataFrame({
    'id': test['id'],
    'Churn': probabilities[0]
})

submission_df.to_csv('submission.csv', index=False)